In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import os
import openpyxl
from pandas import *

In [2]:
dt = 1 / 24
nyears = 1000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Marked Poisson
rain = md.poisson_process(n = int(nsim), alpha = 13.31/1000, lbd = (0.13 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 419.8

shape = np.sin((np.pi*days)/365)                   

daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [3]:
print("Maximum Value is:", max(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 1.8066253767632894
Average value is: 1.1501369863013686


In [4]:
total_rainfall = rain.sum()                  # total rainfall in 'm'
print(f"Total rainfall over years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year in 'mm'
print(f"Average rainfall per year = {RPY:.3f} mm/year")

PET = 850.7
AI = PET/RPY
print(AI)

Total rainfall over years = 625.851 meters
Average rainfall per year = 625.851 mm/year
1.3592697480387328


In [5]:
# Compute TI_DI and TI_Texture

KA_final = 9.0132
n_final = 1.7444
R_max = 3.8                  

def compute_TI(AI):
    AI = np.asarray(AI, dtype=float)
    numerator   = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor

    return np.minimum(TI, 1.0)

# ------------------------------------
# TI FROM SAND (SIGMOID)
# ------------------------------------
sand = 30.0   # JIN sand (%)
s50 = 25.0
k   = 0.08

def TI_from_sand(sand):
    return 1.0 / (1.0 + np.exp(-k * (sand - s50)))

TI_sand = TI_from_sand(sand)
TI_AI = compute_TI(AI)
TI = (TI_sand + TI_AI)/2

print("TI_DI =", TI_AI)
print("TI_tex =", TI_sand)
print("TI Total =", TI)

TI_AI = 0.1960573080472475
TI_sand = 0.598687660112452
TI Total = 0.39737248407984976


In [6]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.2   #m/d,
kstot = 1     #m/d
phitex = 0.45  
phitot = 0.60  

In [7]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████████████████████████████████████████████████████████████████| 8760000/8760000 [20:20<00:00, 7177.34it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       904.498998   89.824211  101.976365  0.100499  0.002943  0.313532   
2       808.981759   55.122763  170.292226  0.083153  0.002151  0.192399   
3       837.439254   49.805694  232.613911  0.091793  0.002316  0.173832   
4       869.225168   65.744531  293.987078  0.094644  0.002595  0.229453   
5       887.182344  107.278951  297.817079  0.098646  0.003013  0.374410   
...            ...         ...         ...       ...       ...       ...   
996    1806.021078  163.283133  217.594643  0.183268  0.005936  0.569896   
997    1806.945738  170.627835  219.236603  0.180103  0.005867  0.595531   
998    1807.633195  169.313810  225.272402  0.171012  0.005568  0.590946   
999    1808.279301  170.167836  221.560652  0.163415  0.005323  0.593931   
1000   1809.008009  172.577809  213.266243  0.179386  0.005852  0.602339   

           

In [8]:
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Jin\JIN.csv", index= True, header = True)
sim_year.to_csv(r"C:\Users\rishabh.singh\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\JIN.csv", index= True, header = True)